# 03. DINOv2: обучение иерархического embedding space

DINOv2 обучается на grayscale crops из UDE, Diatom1042 и Siyue Pu. Gunduz и NII не участвуют в оптимизации классификатора.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
from core.notebook_runtime import bootstrap_notebook, describe_runtime, gpu_preflight, run_guarded

GPU_INDEX = 0
WORKERS = 4
EVAL_BATCH_SIZE = 32
GENERA_PER_BATCH = 4
context = bootstrap_notebook(gpu_index=GPU_INDEX)
PROJECT_ROOT = context.project_root
CONFIG = PROJECT_ROOT / 'configs/classifier.yaml'
CHECKPOINT = PROJECT_ROOT / 'artifacts/classifier/dinov2/best.pt'
RUN_TRAINING = False
INSPECT_CHECKPOINT = False
ENABLE_CLEARML = False
describe_runtime(context)
gpu_preflight(context, minimum_vram_gb=8.0)

## Dataset audit и effective configuration

In [ ]:
from core.config_loader import load_config
overrides = [f'loader.num_workers={WORKERS}', f'loader.eval_batch_size={EVAL_BATCH_SIZE}', f'sampler.genera_per_batch={GENERA_PER_BATCH}']
config = load_config(CONFIG, overrides=overrides)
table_path = PROJECT_ROOT / config['dataset']['table_path']
assert table_path.is_file(), f'Сначала выполните 01_prepare_data.ipynb: {table_path}'
table = pd.read_csv(table_path)
assert not table.empty
assert not table['source'].astype(str).str.lower().eq('nii').any()
display(table.groupby(['split', 'source']).size().rename('crops').reset_index())
display(table.groupby('split').agg(crops=('id_crop', 'size'), genera=('genus', 'nunique'), species=('species', 'nunique')).reset_index())
print(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))

## Обучение

Основная модель — замороженный DINOv2 ViT-B/14 и обучаемая projection head. Метрики retrieval и confusion matrices сохраняются в `artifacts/`; ClearML можно включить после настройки переменных окружения.

In [ ]:
command = context.module_command('scripts.run_train_classifier', '--config', str(CONFIG))
for value in overrides:
    command += ['--set', value]
if ENABLE_CLEARML:
    command += ['--set', 'clearml.enabled=true']
if RUN_TRAINING:
    assert not CHECKPOINT.exists(), f'Checkpoint уже существует: {CHECKPOINT}'
run_guarded(context, command, enabled=RUN_TRAINING, label='dino-train')

## Лучший checkpoint

In [ ]:
if INSPECT_CHECKPOINT:
    import torch
    assert CHECKPOINT.is_file(), CHECKPOINT
    state = torch.load(CHECKPOINT, map_location='cpu')
    print('Epoch:', state.get('epoch'))
    for name, value in sorted(state.get('metrics', {}).items()):
        print(f'{name}: {float(value):.6f}')
else:
    print('Checkpoint inspection skipped')